In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import pybedtools

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from tqdm import tqdm
import multiprocessing as mp
import os


In [ ]:
from utils.tcga_segmentation_workflow import (
    OUT_DIR,
    TRAIN_SAMPLE_ID,
    load_tcga_samples_info,
    select_tcga_samples,
    write_selected_samples_manifest,
    load_tcga_sample_beta_dataframe,
    run_segmentation_for_sample,
    load_meth_ref,
)


In [ ]:
all_samples = load_tcga_samples_info()
all_samples.sample_type.value_counts()

In [ ]:
AUTOSOMES = [f"chr{i}" for i in range(1, 23)]
tumor_type = "TCGA-BRCA"
run_all_matched_tumor_types = True
cv_n_splits = 5
top_n_pmd_regions = 10
max_preload_workers = 70
parallelize_folds = True
max_fold_workers = min(2, cv_n_splits)
max_feature_workers = min(4, os.cpu_count() or 1)
random_region_seed = 42

base_dir = Path("/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/05_tcga_classification_analysis/segmentation")
PMD_SUMMARY_NAME = "segments_PMD.bed"
genome_file = "/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/07_hmms/00_data_prep/data/hg38.genome"


In [ ]:
def summarize_tumor_projects(all_samples, cv_n_splits):
    eligible = all_samples.copy()
    eligible = eligible[eligible["sample_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["project_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["methylation_file"].notna()].copy()
    eligible = eligible[eligible["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])].copy()

    summary = eligible.groupby(["project_id", "sample_type"]).size().unstack(fill_value=0)
    for col in ["Primary Tumor", "Solid Tissue Normal"]:
        if col not in summary.columns:
            summary[col] = 0

    summary = summary[["Primary Tumor", "Solid Tissue Normal"]].reset_index()
    summary = summary.rename(
        columns={
            "Primary Tumor": "n_tumors",
            "Solid Tissue Normal": "n_normals",
        }
    )
    summary["has_matched_normal"] = (summary["n_tumors"] > 0) & (summary["n_normals"] > 0)
    summary["cv_feasible"] = summary["has_matched_normal"] & (summary["n_tumors"] >= cv_n_splits) & (summary["n_normals"] >= cv_n_splits)
    return summary.sort_values(["cv_feasible", "n_normals", "project_id"], ascending=[False, False, True]).reset_index(drop=True)


def get_loop_tumor_types(project_summary_df):
    return project_summary_df.loc[project_summary_df["cv_feasible"], "project_id"].astype(str).tolist()


def select_tumor_vs_normal_cohort(all_samples, tumor_type):
    eligible = all_samples.copy()
    eligible = eligible[eligible["sample_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["project_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["methylation_file"].notna()].copy()
    eligible = eligible[eligible["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])].copy()

    if tumor_type is None:
        cohort = eligible.copy()
    else:
        cohort = eligible[eligible["project_id"].astype(str) == str(tumor_type)].copy()

    cohort = cohort.drop_duplicates(subset=["sample_id"]).reset_index(drop=True)

    n_tumors = int((cohort["sample_type"] == "Primary Tumor").sum())
    n_normals = int((cohort["sample_type"] == "Solid Tissue Normal").sum())
    if n_tumors == 0 or n_normals == 0:
        raise ValueError(f"Cohort {tumor_type!r} does not contain both Primary Tumor and Solid Tissue Normal samples.")

    return cohort


def choose_preload_samples(all_samples, tumor_type, run_all_matched_tumor_types, loop_tumor_types):
    if run_all_matched_tumor_types:
        preload_frames = [select_tumor_vs_normal_cohort(all_samples, project_id) for project_id in loop_tumor_types]
        return pd.concat(preload_frames, ignore_index=True).drop_duplicates(subset=["sample_id"]).reset_index(drop=True)
    return select_tumor_vs_normal_cohort(all_samples, tumor_type)


def build_analysis_label(tumor_type, run_all_matched_tumor_types):
    if run_all_matched_tumor_types:
        return "Looped matched-normal tumor-type comparison"
    if tumor_type is None:
        return "Pan-cancer tumor vs normal"
    return f"{tumor_type} tumor vs normal"

### Analysis mode

This notebook supports **binary tumor-vs-normal analysis only**. You can run a single tumor type such as `TCGA-BRCA`, a pan-cancer tumor-vs-normal analysis with `tumor_type = None`, or loop over tumor types that have matched normals and are feasible for the chosen cross-validation scheme.

All PMD discovery, random-region generation, feature construction, and imputation are performed **inside each training fold only**. This keeps held-out fold samples out of region identification and preprocessing fit steps.

Fold parallelism is optional and can speed up CV, but it increases memory use because each worker needs access to the preloaded methylation matrix and fold-local PMD/random-region state.


In [ ]:
project_summary_df = summarize_tumor_projects(all_samples, cv_n_splits=cv_n_splits)
matched_normal_projects = project_summary_df.loc[project_summary_df["has_matched_normal"], "project_id"].astype(str).tolist()
loop_tumor_types = get_loop_tumor_types(project_summary_df)
excluded_projects_df = project_summary_df.loc[
    project_summary_df["has_matched_normal"] & ~project_summary_df["cv_feasible"],
    ["project_id", "n_tumors", "n_normals"],
].reset_index(drop=True)

if run_all_matched_tumor_types:
    analysis_samples_df = None
    preload_samples_df = choose_preload_samples(all_samples, tumor_type=None, run_all_matched_tumor_types=True, loop_tumor_types=loop_tumor_types)
else:
    analysis_samples_df = select_tumor_vs_normal_cohort(all_samples, tumor_type=tumor_type)
    preload_samples_df = choose_preload_samples(all_samples, tumor_type=tumor_type, run_all_matched_tumor_types=False, loop_tumor_types=[])

analysis_label = build_analysis_label(tumor_type=tumor_type, run_all_matched_tumor_types=run_all_matched_tumor_types)
selected_cohort_summary_df = (
    preload_samples_df.groupby(["project_id", "sample_type"]).size().rename("n_samples").reset_index().sort_values(["project_id", "sample_type"])
)

if not run_all_matched_tumor_types and tumor_type is not None:
    selected_project_row = project_summary_df.loc[project_summary_df["project_id"] == tumor_type]
    if selected_project_row.empty:
        raise ValueError(f"Unknown tumor_type: {tumor_type}")
    if not bool(selected_project_row.iloc[0]["cv_feasible"]):
        raise ValueError(
            f"{tumor_type} has matched normals but is not feasible for {cv_n_splits}-fold CV. "
            f"Tumors={int(selected_project_row.iloc[0]['n_tumors'])}, normals={int(selected_project_row.iloc[0]['n_normals'])}."
        )

In [ ]:
project_summary_df[["project_id", "n_tumors", "n_normals", "has_matched_normal", "cv_feasible"]].head(30), selected_cohort_summary_df.head(30), excluded_projects_df.head(20)

In [ ]:
meth_ref = load_meth_ref()
meth_ref.head()

In [ ]:
probe_df = meth_ref[["CpG_chrm", "CpG_beg", "CpG_end", "key"]].copy()
probe_df = probe_df.dropna(subset=["CpG_chrm", "CpG_beg", "CpG_end"])
probe_df["CpG_beg"] = probe_df["CpG_beg"].astype(int)
probe_df["CpG_end"] = probe_df["CpG_end"].astype(int)
probe_df = probe_df[probe_df["CpG_end"] > probe_df["CpG_beg"]]
probe_df = probe_df.rename(columns={
    "CpG_chrm": "chrom",
    "CpG_beg": "start",
    "CpG_end": "end",
    "key": "name",
})
probe_df = probe_df[probe_df["chrom"].isin(AUTOSOMES)].copy()
probes_bed = pybedtools.BedTool.from_dataframe(probe_df[["chrom", "start", "end", "name"]])
probe_df.head()

In [ ]:
def _load_one_sample(sample_id, meth_file):
    meth_data = load_tcga_sample_beta_dataframe(sample_id, meth_file, drop_nas=False)
    return sample_id, meth_data


def _load_one_sample_star(args):
    return _load_one_sample(*args)


def preload_all_samples_methylation_data(samples_df, meth_ref, max_workers=None):
    coord_cols = ["CpG_chrm", "CpG_beg", "CpG_end"]
    coord_df = meth_ref.loc[:, ["key", *coord_cols]].copy()
    coord_df["CpG_chrm"] = coord_df["CpG_chrm"].astype(str)
    if not coord_df["CpG_chrm"].str.startswith("chr").all():
        coord_df["CpG_chrm"] = "chr" + coord_df["CpG_chrm"].str.replace("^chr", "", regex=True)
    coord_df["CpG_beg"] = pd.to_numeric(coord_df["CpG_beg"], errors="coerce")
    coord_df["CpG_end"] = pd.to_numeric(coord_df["CpG_end"], errors="coerce")
    coord_df = coord_df.set_index("key")
    sample_beta_by_id = {}

    if max_workers is None:
        max_workers = min(len(samples_df), os.cpu_count() or 1)

    sample_jobs = [
        (str(sample_row.sample_id), sample_row.methylation_file)
        for sample_row in samples_df.itertuples(index=False)
    ]

    ctx = mp.get_context("fork") if os.name != "nt" else mp.get_context()
    with ctx.Pool(processes=max_workers) as pool:
        with tqdm(total=len(sample_jobs), desc="Loading methylation data") as pbar:
            for returned_sample_id, meth_data in pool.imap_unordered(_load_one_sample_star, sample_jobs):
                sample_beta = meth_data.set_index("probe")["beta"]
                sample_beta_by_id[returned_sample_id] = sample_beta.reindex(coord_df.index)
                pbar.update(1)

    sample_beta_df = pd.DataFrame(sample_beta_by_id)
    meth_data_df = pd.concat([coord_df, sample_beta_df], axis=1).reset_index()
    return meth_data_df


def ensure_meth_data_df_has_samples(existing_meth_data_df, samples_df, meth_ref, max_workers=None):
    required_sample_ids = samples_df["sample_id"].astype(str).tolist()
    existing_sample_ids = {str(col) for col in existing_meth_data_df.columns if str(col) not in {"key", "CpG_chrm", "CpG_beg", "CpG_end"}}
    missing_sample_ids = [sample_id for sample_id in required_sample_ids if sample_id not in existing_sample_ids]
    if not missing_sample_ids:
        return existing_meth_data_df

    print(f"Loading {len(missing_sample_ids)} missing sample columns into meth_data_df")
    missing_samples_df = samples_df.loc[samples_df["sample_id"].astype(str).isin(missing_sample_ids)].copy()
    missing_meth_data_df = preload_all_samples_methylation_data(missing_samples_df, meth_ref, max_workers=max_workers)

    existing_indexed = existing_meth_data_df.set_index("key")
    missing_indexed = missing_meth_data_df.set_index("key")
    missing_value_cols = [
        col for col in missing_indexed.columns
        if col not in {"CpG_chrm", "CpG_beg", "CpG_end"}
    ]
    combined = pd.concat([existing_indexed, missing_indexed[missing_value_cols]], axis=1)
    return combined.reset_index()


try:
    meth_data_df
except NameError:
    meth_data_df = preload_all_samples_methylation_data(preload_samples_df, meth_ref, max_workers=max_preload_workers)
else:
    meth_data_df = ensure_meth_data_df_has_samples(meth_data_df, preload_samples_df, meth_ref, max_workers=max_preload_workers)


In [ ]:
meth_data_df

In [ ]:
METRIC_COLS = [
    "mcc",
    "balanced_accuracy",
    "macro_f1",
    "specificity",
    "negative_precision",
]


def check_for_missing(meth_data_df, chrom, start, end, min_cpgs=1):
    subset = meth_data_df[
        (meth_data_df["CpG_chrm"].astype(str) == str(chrom))
        & (meth_data_df["CpG_beg"] >= start)
        & (meth_data_df["CpG_end"] <= end)
    ]

    if subset.empty:
        return True

    beta_cols = [
        c for c in meth_data_df.columns if c not in {"CpG_chrm", "CpG_beg", "CpG_end", "key"}
    ]
    if subset[beta_cols].isna().all(axis=0).any():
        return True
    if len(subset) < min_cpgs:
        return True
    return False


def subtract_used_span(df, chrom, used_start, used_end):
    kept = []
    for row in df.itertuples(index=False):
        if row.chr != chrom or row.end <= used_start or row.start >= used_end:
            kept.append({"chr": row.chr, "start": int(row.start), "end": int(row.end)})
            continue
        if row.start < used_start:
            kept.append({"chr": row.chr, "start": int(row.start), "end": int(used_start)})
        if row.end > used_end:
            kept.append({"chr": row.chr, "start": int(used_end), "end": int(row.end)})
    out = pd.DataFrame(kept)
    if not out.empty:
        out = out[out["end"] > out["start"]].reset_index(drop=True)
    return out


def prepare_random_region_pool(non_pmd_regions_df):
    pool = non_pmd_regions_df.copy().reset_index(drop=True)
    pool["chr"] = pool["chr"].astype(str)
    pool["length"] = pool["end"] - pool["start"]
    return pool[pool["chr"].isin(AUTOSOMES)].reset_index(drop=True)


def _sample_interval_from_row(row, interval_len, meth_data_df, min_cpgs, rng, max_tries):
    chrom = str(row["chr"])
    region_len = int(row["length"])
    interval_len = int(interval_len)
    max_offset = region_len - interval_len
    if max_offset < 0:
        return None

    for _ in range(max_tries):
        offset = 0 if max_offset == 0 else rng.randint(0, max_offset)
        used_start = int(row["start"] + offset)
        used_end = used_start + interval_len
        if not check_for_missing(meth_data_df, chrom, used_start, used_end, min_cpgs=min_cpgs):
            return {
                "chr": chrom,
                "start": used_start,
                "end": used_end,
                "requested_length": int(interval_len),
                "realized_length": int(interval_len),
                "pool_row_length": region_len,
            }
    return None


def sample_non_pmd_interval(pool, meth_data_df, requested_len, min_cpgs, rng, max_tries=200):
    if pool.empty:
        return None

    candidate_idx = list(pool.index)
    rng.shuffle(candidate_idx)

    exact_or_longer = [idx for idx in candidate_idx if int(pool.loc[idx, "length"]) >= requested_len]
    for idx in exact_or_longer:
        row = pool.loc[idx]
        chosen = _sample_interval_from_row(row, requested_len, meth_data_df, min_cpgs, rng, max_tries)
        if chosen is not None:
            chosen["match_type"] = "exact_length"
            chosen["pool_row_index"] = int(idx)
            return chosen

    shorter_idx = [idx for idx in candidate_idx if 0 < int(pool.loc[idx, "length"]) < requested_len]
    shorter_idx = sorted(shorter_idx, key=lambda idx: int(pool.loc[idx, "length"]), reverse=True)
    for idx in shorter_idx:
        row = pool.loc[idx]
        shorter_len = int(row["length"])
        chosen = _sample_interval_from_row(row, shorter_len, meth_data_df, min_cpgs, rng, max_tries)
        if chosen is not None:
            chosen["match_type"] = "shortened_interval"
            chosen["pool_row_index"] = int(idx)
            return chosen
    return None


def get_random_regions(pmd_regions_df, non_pmd_regions_df, meth_data_df, seed=42, min_cpgs=1):
    rng = random.Random(seed)
    random_regions = []
    skipped = []
    pool = prepare_random_region_pool(non_pmd_regions_df)

    for pmd_row in pmd_regions_df.itertuples(index=False):
        requested_len = int(pmd_row.end - pmd_row.start)
        chosen = sample_non_pmd_interval(pool, meth_data_df, requested_len, min_cpgs, rng)
        if chosen is None:
            skipped.append({
                "chr": pmd_row.chr,
                "start": int(pmd_row.start),
                "end": int(pmd_row.end),
                "requested_length": requested_len,
                "seed": seed,
                "reason": "no PMD-free interval with enough CpG coverage could be placed",
            })
            continue

        pool = subtract_used_span(pool[["chr", "start", "end"]], chosen["chr"], chosen["start"], chosen["end"])
        if not pool.empty:
            pool = prepare_random_region_pool(pool)

        random_regions.append({
            "chr": chosen["chr"],
            "start": chosen["start"],
            "end": chosen["end"],
            "match_type": chosen["match_type"],
            "requested_length": requested_len,
            "realized_length": chosen["realized_length"],
            "length_ratio": chosen["realized_length"] / requested_len,
            "seed": seed,
        })

    return pd.DataFrame(random_regions), pd.DataFrame(skipped), pool


def validate_random_regions(random_regions_df, non_pmd_regions_df, pmd_union_bed, genome_file):
    required_cols = {"chr", "start", "end", "requested_length", "realized_length", "match_type", "seed"}
    if random_regions_df.empty:
        return pd.DataFrame(columns=["check", "passed", "detail"])

    missing_cols = required_cols.difference(random_regions_df.columns)
    if missing_cols:
        raise ValueError(f"random_regions_df is missing required columns: {sorted(missing_cols)}")

    autosomes_ok = random_regions_df["chr"].isin(AUTOSOMES).all()
    lengths_ok = (random_regions_df["realized_length"] <= random_regions_df["requested_length"]).all()
    containment_ok = random_regions_df.apply(
        lambda row: ((non_pmd_regions_df["chr"] == row["chr"]) & (non_pmd_regions_df["start"] <= row["start"]) & (non_pmd_regions_df["end"] >= row["end"])).any(),
        axis=1,
    ).all()

    random_bed = pybedtools.BedTool.from_dataframe(random_regions_df[["chr", "start", "end"]]).sort(g=genome_file)
    overlap_df = random_bed.intersect(pmd_union_bed, u=True).to_dataframe(names=["chr", "start", "end"])
    overlap_ok = overlap_df.empty

    checks = pd.DataFrame([
        {"check": "autosomes_only", "passed": bool(autosomes_ok), "detail": int((~random_regions_df["chr"].isin(AUTOSOMES)).sum())},
        {"check": "contained_within_single_non_pmd_interval", "passed": bool(containment_ok), "detail": int(len(random_regions_df) - random_regions_df.apply(lambda row: ((non_pmd_regions_df["chr"] == row["chr"]) & (non_pmd_regions_df["start"] <= row["start"]) & (non_pmd_regions_df["end"] >= row["end"])).any(), axis=1).sum())},
        {"check": "no_pmd_overlap", "passed": bool(overlap_ok), "detail": int(len(overlap_df))},
        {"check": "realized_length_lte_requested_length", "passed": bool(lengths_ok), "detail": int((random_regions_df["realized_length"] > random_regions_df["requested_length"]).sum())},
    ])

    failed = checks[~checks["passed"]]
    if not failed.empty:
        raise ValueError(f"Random-region validation failed: {failed.to_dict(orient='records')}")

    return checks


def get_pmd_summary_path(sample_id, base_dir, summary_name=PMD_SUMMARY_NAME):
    return Path(base_dir) / "methylseg" / str(sample_id) / "out" / "hm450k" / "summary_files" / summary_name


def load_pmd_regions_for_samples(sample_ids, base_dir, summary_name=PMD_SUMMARY_NAME):
    all_pmds = []
    missing_paths = []
    for sample_id in sample_ids:
        summary_file = get_pmd_summary_path(sample_id, base_dir=base_dir, summary_name=summary_name)
        if not summary_file.exists():
            missing_paths.append(str(summary_file))
            continue
        try:
            pmd_df = pd.read_csv(summary_file, sep="	", names=["chr", "start", "end", "RegionType"])
        except pd.errors.EmptyDataError:
            pmd_df = pd.DataFrame(columns=["chr", "start", "end", "RegionType"])
        if pmd_df.empty:
            continue
        pmd_df["sample_id"] = str(sample_id)
        all_pmds.append(pmd_df)

    if all_pmds:
        all_pmds_df = pd.concat(all_pmds, ignore_index=True)
        all_pmds_df = all_pmds_df[all_pmds_df["chr"].isin(AUTOSOMES)].copy()
    else:
        all_pmds_df = pd.DataFrame(columns=["chr", "start", "end", "RegionType", "sample_id"])

    return all_pmds_df, missing_paths


def build_fold_region_sets(train_cancer_samples_df, probe_df, probes_bed, base_dir, genome_file, top_n_pmd_regions):
    all_pmds_df, missing_paths = load_pmd_regions_for_samples(train_cancer_samples_df["sample_id"].astype(str).tolist(), base_dir=base_dir)
    if all_pmds_df.empty:
        raise ValueError("No PMD summary rows were available for the training tumor samples in this fold.")

    all_pmds_bed = pybedtools.BedTool.from_dataframe(all_pmds_df[["chr", "start", "end", "sample_id"]]).sort(g=genome_file)
    all_pmds_bed = all_pmds_bed.merge(c=4, o="distinct")
    all_pmds_bed_df = all_pmds_bed.to_dataframe(names=["chr", "start", "end", "sample_ids"])
    all_pmds_bed_df["n_samples"] = all_pmds_bed_df["sample_ids"].str.split(",").apply(len)
    all_pmds_bed_df["length"] = all_pmds_bed_df["end"] - all_pmds_bed_df["start"]
    all_pmds_bed_df = all_pmds_bed_df[(all_pmds_bed_df["length"] >= 1000) & (all_pmds_bed_df["length"] <= 100000000)].copy()
    if all_pmds_bed_df.empty:
        raise ValueError("No merged PMD regions passed the length filters in this fold.")

    top_shared_pmds = all_pmds_bed_df.sort_values(["n_samples", "length"], ascending=[False, False]).head(top_n_pmd_regions).reset_index(drop=True)
    if top_shared_pmds.empty:
        raise ValueError("No top shared PMD regions were available after filtering in this fold.")

    all_pmds_union_bed = pybedtools.BedTool.from_dataframe(all_pmds_bed_df[["chr", "start", "end"]]).sort(g=genome_file)
    non_region_bed = all_pmds_union_bed.complement(g=genome_file)
    regions_with_probes = non_region_bed.intersect(probes_bed, u=True)
    non_pmd_regions_df = regions_with_probes.to_dataframe(names=["chr", "start", "end"])
    if not non_pmd_regions_df.empty:
        non_pmd_regions_df = non_pmd_regions_df[non_pmd_regions_df["chr"].isin(AUTOSOMES)].copy().reset_index(drop=True)

    if non_pmd_regions_df.empty:
        raise ValueError("No non-PMD intervals with probe support were available in this fold.")

    metadata = {
        "n_train_cancer_samples": int(train_cancer_samples_df["sample_id"].nunique()),
        "n_merged_pmd_regions": int(len(all_pmds_bed_df)),
        "n_top_pmd_regions": int(len(top_shared_pmds)),
        "n_non_pmd_regions": int(len(non_pmd_regions_df)),
        "n_missing_pmd_files": int(len(missing_paths)),
    }
    return top_shared_pmds, all_pmds_union_bed, non_pmd_regions_df, metadata

In [ ]:
def calculate_average_beta_in_region(meth_data, region):
    chrom, start, end = region
    region_meth_data = meth_data.loc[
        (meth_data["CpG_chrm"] == chrom)
        & (meth_data["CpG_beg"] >= start)
        & (meth_data["CpG_end"] <= end)
    ]
    if region_meth_data.empty:
        return None
    return region_meth_data["beta"].mean()


def get_mp_context():
    return mp.get_context("fork") if os.name != "nt" else mp.get_context()


def resolve_worker_count(max_workers, n_tasks):
    if max_workers is None:
        max_workers = os.cpu_count() or 1
    return max(1, min(int(max_workers), int(max(1, n_tasks))))


_FEATURE_EXTRACTION_CONTEXT = {}
_BINARY_FOLD_CONTEXT = {}


def register_mp_callable(func):
    try:
        main_mod = __import__("__main__")
        setattr(main_mod, func.__name__, func)
        func.__module__ = "__main__"
    except Exception:
        pass
    return func


def configure_feature_extraction_context(meth_data_df, region_specs):
    global _FEATURE_EXTRACTION_CONTEXT
    _FEATURE_EXTRACTION_CONTEXT = {
        "meth_data_df": meth_data_df,
        "region_specs": list(region_specs),
    }


def _compute_sample_feature_row(sample_id):
    meth_data_df = _FEATURE_EXTRACTION_CONTEXT["meth_data_df"]
    region_specs = _FEATURE_EXTRACTION_CONTEXT["region_specs"]
    meth_data = meth_data_df.loc[:, ["CpG_chrm", "CpG_beg", "CpG_end", sample_id]].rename(columns={sample_id: "beta"})
    sample_features = {}
    for feature_name, chrom, start, end in region_specs:
        sample_features[feature_name] = calculate_average_beta_in_region(meth_data, (chrom, start, end))
    return sample_id, sample_features


register_mp_callable(_compute_sample_feature_row)


def create_ml_input(regions_df, samples_df, meth_data_df, max_workers=1):
    if regions_df.empty:
        raise ValueError("regions_df is empty, so no features can be created.")

    region_specs = [(row.Index, row.chr, row.start, row.end) for row in regions_df.itertuples()]
    sample_ids = samples_df["sample_id"].astype(str).tolist()
    feature_rows = {}

    worker_count = resolve_worker_count(max_workers=max_workers, n_tasks=len(sample_ids))
    if worker_count > 1 and len(sample_ids) > 1:
        configure_feature_extraction_context(meth_data_df, region_specs)
        with get_mp_context().Pool(processes=worker_count) as pool:
            for sample_id, sample_features in pool.imap_unordered(_compute_sample_feature_row, sample_ids):
                feature_rows[sample_id] = sample_features
    else:
        configure_feature_extraction_context(meth_data_df, region_specs)
        for sample_id in sample_ids:
            returned_sample_id, sample_features = _compute_sample_feature_row(sample_id)
            feature_rows[returned_sample_id] = sample_features

    X = pd.DataFrame.from_dict(feature_rows, orient="index")
    X.index.name = "sample_id"
    X = X.dropna(axis=1, how="all")
    if X.shape[1] == 0:
        raise ValueError("No usable region features were created. All candidate regions were missing CpGs across all samples.")

    y = samples_df.set_index("sample_id").loc[X.index, "sample_type"].replace({"Primary Tumor": "Cancer", "Metastatic": "Cancer"})
    return X, y


def prepare_train_test_data(X, y, train_sample_ids, test_sample_ids):
    if X.empty or X.shape[1] == 0:
        raise ValueError("X has no feature columns to train on.")

    train_sample_ids = [sample_id for sample_id in train_sample_ids if sample_id in X.index]
    test_sample_ids = [sample_id for sample_id in test_sample_ids if sample_id in X.index]

    X_train = X.loc[train_sample_ids]
    X_test = X.loc[test_sample_ids]
    y_train = y.loc[train_sample_ids]
    y_test = y.loc[test_sample_ids]

    X_train = X_train.dropna(axis=1, how="all")
    if X_train.shape[1] == 0:
        raise ValueError("All training feature columns are entirely missing after filtering, so classification cannot run.")

    X_test = X_test.loc[:, X_train.columns]
    imputer = SimpleImputer(strategy="mean")
    X_train = pd.DataFrame(imputer.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test = pd.DataFrame(imputer.transform(X_test), index=X_test.index, columns=X_test.columns)
    return X_train, X_test, y_train, y_test


def compute_binary_negative_metrics(y_true, y_pred, negative_label="Solid Tissue Normal"):
    labels = [negative_label, "Cancer"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if cm.shape != (2, 2):
        return 0.0, 0.0
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    negative_precision = tn / (tn + fn) if (tn + fn) else 0.0
    return float(specificity), float(negative_precision)


def summarize_classification(y_true, y_pred, analysis, n_regions, n_shortened=0, mean_length_ratio=np.nan):
    _, _, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    specificity, negative_precision = compute_binary_negative_metrics(y_true, y_pred)
    return {
        "analysis": analysis,
        "n_regions": int(n_regions),
        "n_shortened": int(n_shortened),
        "mean_length_ratio": float(mean_length_ratio) if not pd.isna(mean_length_ratio) else np.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(macro_f1),
        "specificity": specificity,
        "negative_precision": negative_precision,
    }


def run_ml_classification(X, y, train_sample_ids, test_sample_ids, analysis, class_weight="balanced", region_metadata=None):
    X_train, X_test, y_train, y_test = prepare_train_test_data(X, y, train_sample_ids, test_sample_ids)
    rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight=class_weight)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    region_metadata = region_metadata if region_metadata is not None else pd.DataFrame()
    metrics = summarize_classification(
        y_true=y_test,
        y_pred=y_pred,
        analysis=analysis,
        n_regions=X_train.shape[1],
        n_shortened=region_metadata.get("match_type", pd.Series(dtype=object)).eq("shortened_interval").sum() if not region_metadata.empty else 0,
        mean_length_ratio=region_metadata.get("length_ratio", pd.Series(dtype=float)).mean() if not region_metadata.empty else np.nan,
    )
    return metrics


def run_always_cancer_baseline(y, train_sample_ids, test_sample_ids, analysis="Always cancer"):
    _, _, _, y_test = prepare_train_test_data(
        X=pd.DataFrame(index=y.index, data={"dummy": np.zeros(len(y))}),
        y=y,
        train_sample_ids=train_sample_ids,
        test_sample_ids=test_sample_ids,
    )
    y_pred = pd.Series("Cancer", index=y_test.index)
    return summarize_classification(y_true=y_test, y_pred=y_pred, analysis=analysis, n_regions=0)


def summarize_fold_results(fold_results_df, metadata):
    rows = []
    for analysis, analysis_df in fold_results_df.groupby("analysis", sort=False):
        row = {
            "tumor_type": metadata["tumor_type"],
            "cohort_label": metadata["cohort_label"],
            "analysis": analysis,
            "n_samples": metadata["n_samples"],
            "n_tumors": metadata["n_tumors"],
            "n_normals": metadata["n_normals"],
            "n_projects": metadata["n_projects"],
            "n_folds_completed": int(analysis_df["fold"].nunique()),
            "mean_n_regions": analysis_df["n_regions"].mean(),
            "mean_n_shortened": analysis_df["n_shortened"].mean(),
            "mean_length_ratio": analysis_df["mean_length_ratio"].mean(),
        }
        for metric in METRIC_COLS:
            row[f"{metric}_mean"] = analysis_df[metric].mean()
            row[f"{metric}_std"] = analysis_df[metric].std()
            row[f"{metric}_min"] = analysis_df[metric].min()
            row[f"{metric}_median"] = analysis_df[metric].median()
            row[f"{metric}_max"] = analysis_df[metric].max()
        rows.append(row)
    return pd.DataFrame(rows)


def aggregate_overall_metrics(per_tumor_summary_df, weight_col="n_samples"):
    rows = []
    for analysis, analysis_df in per_tumor_summary_df.groupby("analysis", sort=False):
        for aggregation_type in ["macro", "weighted"]:
            row = {"analysis": analysis, "aggregation_type": aggregation_type, "n_tumor_types": int(analysis_df["tumor_type"].nunique())}
            for metric in METRIC_COLS:
                metric_col = f"{metric}_mean"
                if aggregation_type == "macro":
                    row[metric] = analysis_df[metric_col].mean()
                else:
                    row[metric] = np.average(analysis_df[metric_col], weights=analysis_df[weight_col])
            rows.append(row)
    return pd.DataFrame(rows)


def plot_cv_analysis_results(fold_results_df, cohort_label):
    metrics_to_plot = ["mcc", "balanced_accuracy", "macro_f1", "specificity", "negative_precision"]
    analysis_order = ["PMD", "Random", "Always cancer"]
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(20, 5))

    for ax, metric in zip(axes, metrics_to_plot):
        data = [fold_results_df.loc[fold_results_df["analysis"] == analysis, metric].dropna() for analysis in analysis_order]
        ax.boxplot(
            data,
            patch_artist=True,
            labels=analysis_order,
            boxprops={"facecolor": "lightgray", "edgecolor": "dimgray", "linewidth": 1.5},
            medianprops={"color": "black", "linewidth": 2},
            whiskerprops={"color": "dimgray", "linewidth": 1.5},
            capprops={"color": "dimgray", "linewidth": 1.5},
            flierprops={"marker": "o", "markerfacecolor": "gray", "markeredgecolor": "dimgray", "alpha": 0.6, "markersize": 4},
        )
        ax.set_title(metric.replace("_", " ").title())
        ax.set_ylabel("Score")
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
    fig.suptitle(f"Fold-level CV scores: {cohort_label}", y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


def plot_loop_overall_summary(overall_summary_df, cohort_label):
    metrics_to_plot = ["mcc", "balanced_accuracy", "macro_f1", "specificity", "negative_precision"]
    analysis_order = ["PMD", "Random", "Always cancer"]
    aggregation_order = ["macro", "weighted"]
    colors = {"PMD": "#4C78A8", "Random": "#A0A0A0", "Always cancer": "#E45756"}
    x = np.arange(len(metrics_to_plot))
    width = 0.24
    fig, axes = plt.subplots(1, len(aggregation_order), figsize=(14, 5), sharey=True)

    if len(aggregation_order) == 1:
        axes = [axes]

    for ax, aggregation_type in zip(axes, aggregation_order):
        subset = overall_summary_df.loc[overall_summary_df["aggregation_type"] == aggregation_type].copy()
        for idx, analysis in enumerate(analysis_order):
            analysis_row = subset.loc[subset["analysis"] == analysis]
            if analysis_row.empty:
                values = [np.nan] * len(metrics_to_plot)
            else:
                values = [float(analysis_row.iloc[0][metric]) for metric in metrics_to_plot]
            ax.bar(x + (idx - 1) * width, values, width=width, label=analysis, color=colors[analysis], alpha=0.9)

        ax.set_xticks(x)
        ax.set_xticklabels([metric.replace("_", " ").title() for metric in metrics_to_plot], rotation=20)
        ax.set_ylim(0, 1.02)
        ax.set_title(f"{aggregation_type.title()} average")
        ax.set_ylabel("Score")
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(analysis_order), frameon=False)
    fig.suptitle(f"Averaged cross-tumor CV scores: {cohort_label}", y=1.05)
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


def build_conclusion(summary_df):
    pmd_row = summary_df.loc[summary_df["analysis"] == "PMD"]
    random_row = summary_df.loc[summary_df["analysis"] == "Random"]
    if pmd_row.empty or random_row.empty:
        return "Evidence is inconclusive because one or more comparison analyses did not produce summary rows."

    pmd_mcc = float(pmd_row.iloc[0]["mcc_mean"])
    pmd_bal = float(pmd_row.iloc[0]["balanced_accuracy_mean"])
    rand_mcc = float(random_row.iloc[0]["mcc_mean"])
    rand_bal = float(random_row.iloc[0]["balanced_accuracy_mean"])
    if pmd_mcc > rand_mcc and pmd_bal > rand_bal:
        return "PMD regions outperform the PMD-free random baseline on mean MCC and mean balanced accuracy across folds."
    return "Evidence is weak or inconclusive: PMD regions do not clearly outperform the PMD-free random baseline on the primary imbalance-aware metrics."


def configure_binary_fold_context(context):
    global _BINARY_FOLD_CONTEXT
    _BINARY_FOLD_CONTEXT = context


def _run_binary_fold_task(task):
    context = _BINARY_FOLD_CONTEXT
    fold_idx = int(task["fold"])
    train_idx = task["train_idx"]
    test_idx = task["test_idx"]
    analysis_samples_df = context["analysis_samples_df"]
    cohort_label = context["cohort_label"]
    tumor_type = context["tumor_type"]
    feature_workers = context["feature_workers"]

    train_df = analysis_samples_df.iloc[train_idx].reset_index(drop=True)
    test_df = analysis_samples_df.iloc[test_idx].reset_index(drop=True)
    train_sample_ids = train_df["sample_id"].astype(str).tolist()
    test_sample_ids = test_df["sample_id"].astype(str).tolist()
    train_cancer_samples_df = train_df[train_df["sample_type"] == "Primary Tumor"].copy()

    warnings = []
    fold_results = []
    audit_frames = []

    try:
        top_shared_pmds, all_pmds_union_bed, non_pmd_regions_df, region_meta = build_fold_region_sets(
            train_cancer_samples_df=train_cancer_samples_df,
            probe_df=context["probe_df"],
            probes_bed=context["probes_bed"],
            base_dir=context["base_dir"],
            genome_file=context["genome_file"],
            top_n_pmd_regions=context["top_n_pmd_regions"],
        )
    except ValueError as exc:
        warnings.append({"tumor_type": tumor_type, "fold": fold_idx, "warning": str(exc)})
        return {"fold": fold_idx, "fold_results": fold_results, "audit_df": pd.DataFrame(), "warnings": warnings}

    X_pmd, y_binary = create_ml_input(
        top_shared_pmds,
        analysis_samples_df,
        context["meth_data_df"],
        max_workers=feature_workers,
    )
    pmd_metrics = run_ml_classification(X_pmd, y_binary, train_sample_ids, test_sample_ids, analysis="PMD")
    pmd_metrics.update({
        "fold": fold_idx,
        "tumor_type": tumor_type,
        "cohort_label": cohort_label,
        "n_train_samples": len(train_df),
        "n_test_samples": len(test_df),
        "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
        "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
    })
    pmd_metrics.update(region_meta)
    fold_results.append(pmd_metrics)

    baseline_metrics = run_always_cancer_baseline(y_binary, train_sample_ids, test_sample_ids)
    baseline_metrics.update({
        "fold": fold_idx,
        "tumor_type": tumor_type,
        "cohort_label": cohort_label,
        "n_train_samples": len(train_df),
        "n_test_samples": len(test_df),
        "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
        "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
    })
    baseline_metrics.update(region_meta)
    fold_results.append(baseline_metrics)

    random_regions_df, skipped_df, _ = get_random_regions(
        pmd_regions_df=top_shared_pmds,
        non_pmd_regions_df=non_pmd_regions_df,
        meth_data_df=context["meth_data_df"],
        seed=context["random_region_seed"] + fold_idx,
    )

    if random_regions_df.empty:
        warnings.append({
            "tumor_type": tumor_type,
            "fold": fold_idx,
            "warning": "No PMD-free random regions could be generated for this fold.",
        })
    else:
        validation_df = validate_random_regions(
            random_regions_df=random_regions_df,
            non_pmd_regions_df=non_pmd_regions_df,
            pmd_union_bed=all_pmds_union_bed,
            genome_file=context["genome_file"],
        )
        validation_df["fold"] = fold_idx
        validation_df["tumor_type"] = tumor_type
        audit_frames.append(validation_df)

        X_random, y_random = create_ml_input(
            random_regions_df,
            analysis_samples_df,
            context["meth_data_df"],
            max_workers=feature_workers,
        )
        random_metrics = run_ml_classification(
            X_random,
            y_random,
            train_sample_ids,
            test_sample_ids,
            analysis="Random",
            region_metadata=random_regions_df,
        )
        random_metrics.update({
            "fold": fold_idx,
            "tumor_type": tumor_type,
            "cohort_label": cohort_label,
            "n_train_samples": len(train_df),
            "n_test_samples": len(test_df),
            "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
            "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
            "n_random_skipped": int(len(skipped_df)),
        })
        random_metrics.update(region_meta)
        fold_results.append(random_metrics)

        if (random_regions_df["match_type"] == "shortened_interval").any():
            warnings.append({
                "tumor_type": tumor_type,
                "fold": fold_idx,
                "warning": "Some random regions were shortened to stay PMD-free within a single complement interval.",
            })

    if not random_regions_df.empty and len(random_regions_df) < len(top_shared_pmds):
        warnings.append({
            "tumor_type": tumor_type,
            "fold": fold_idx,
            "warning": "Fewer PMD-free random regions were available than PMD regions in this fold.",
        })

    audit_df = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
    return {"fold": fold_idx, "fold_results": fold_results, "audit_df": audit_df, "warnings": warnings}


register_mp_callable(_run_binary_fold_task)


def run_tumor_vs_normal_analysis(analysis_samples_df, cohort_label, tumor_type, cv_n_splits, meth_data_df, probe_df, probes_bed, base_dir, genome_file, top_n_pmd_regions, random_region_seed, parallelize_folds=True, max_fold_workers=1, max_feature_workers=1):
    labels = np.where(analysis_samples_df["sample_type"].astype(str) == "Solid Tissue Normal", "Solid Tissue Normal", "Cancer")
    splitter = StratifiedKFold(n_splits=cv_n_splits, shuffle=True, random_state=42)
    split_tasks = [
        {"fold": fold_idx, "train_idx": train_idx, "test_idx": test_idx}
        for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(analysis_samples_df, labels), start=1)
    ]

    fold_worker_count = resolve_worker_count(max_workers=max_fold_workers, n_tasks=len(split_tasks))
    use_parallel_folds = bool(parallelize_folds and fold_worker_count > 1 and len(split_tasks) > 1)
    feature_worker_count = 1 if use_parallel_folds else resolve_worker_count(max_workers=max_feature_workers, n_tasks=len(analysis_samples_df))

    fold_context = {
        "analysis_samples_df": analysis_samples_df,
        "cohort_label": cohort_label,
        "tumor_type": tumor_type,
        "meth_data_df": meth_data_df,
        "probe_df": probe_df,
        "probes_bed": probes_bed,
        "base_dir": base_dir,
        "genome_file": genome_file,
        "top_n_pmd_regions": top_n_pmd_regions,
        "random_region_seed": random_region_seed,
        "feature_workers": feature_worker_count,
    }

    configure_binary_fold_context(fold_context)
    fold_outputs = []
    if use_parallel_folds:
        with get_mp_context().Pool(processes=fold_worker_count) as pool:
            for fold_output in tqdm(pool.imap_unordered(_run_binary_fold_task, split_tasks), total=len(split_tasks), desc=f"{cohort_label} folds"):
                fold_outputs.append(fold_output)
    else:
        for task in tqdm(split_tasks, total=len(split_tasks), desc=f"{cohort_label} folds"):
            fold_outputs.append(_run_binary_fold_task(task))

    fold_outputs = sorted(fold_outputs, key=lambda item: item["fold"])
    fold_results = []
    audit_frames = []
    warnings = []
    for fold_output in fold_outputs:
        fold_results.extend(fold_output["fold_results"])
        if not fold_output["audit_df"].empty:
            audit_frames.append(fold_output["audit_df"])
        warnings.extend(fold_output["warnings"])

    fold_results_df = pd.DataFrame(fold_results)
    if fold_results_df.empty:
        raise ValueError(f"No fold-level results could be generated for cohort {cohort_label}.")

    metadata = {
        "tumor_type": tumor_type if tumor_type is not None else "PAN_CANCER",
        "cohort_label": cohort_label,
        "n_samples": int(analysis_samples_df["sample_id"].nunique()),
        "n_tumors": int((analysis_samples_df["sample_type"] == "Primary Tumor").sum()),
        "n_normals": int((analysis_samples_df["sample_type"] == "Solid Tissue Normal").sum()),
        "n_projects": int(analysis_samples_df["project_id"].nunique()),
        "n_folds": cv_n_splits,
        "runtime_parallelize_folds": bool(use_parallel_folds),
        "runtime_max_fold_workers": int(fold_worker_count),
        "runtime_max_feature_workers": int(feature_worker_count),
    }
    audit_df = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
    summary_df = summarize_fold_results(fold_results_df, metadata)
    return fold_results_df, audit_df, summary_df, warnings, metadata


def run_tumor_type_loop(all_samples, tumor_types, cv_n_splits, meth_data_df, probe_df, probes_bed, base_dir, genome_file, top_n_pmd_regions, random_region_seed, parallelize_folds=True, max_fold_workers=1, max_feature_workers=1):
    fold_frames = []
    audit_frames = []
    summary_frames = []
    warning_records = []

    for project_id in tumor_types:
        cohort_df = select_tumor_vs_normal_cohort(all_samples, tumor_type=project_id)
        cohort_label = f"{project_id} tumor vs normal"
        fold_df, audit_df, summary_df, warnings, metadata = run_tumor_vs_normal_analysis(
            analysis_samples_df=cohort_df,
            cohort_label=cohort_label,
            tumor_type=project_id,
            cv_n_splits=cv_n_splits,
            meth_data_df=meth_data_df,
            probe_df=probe_df,
            probes_bed=probes_bed,
            base_dir=base_dir,
            genome_file=genome_file,
            top_n_pmd_regions=top_n_pmd_regions,
            random_region_seed=random_region_seed,
            parallelize_folds=parallelize_folds,
            max_fold_workers=max_fold_workers,
            max_feature_workers=max_feature_workers,
        )
        fold_frames.append(fold_df)
        if not audit_df.empty:
            audit_frames.append(audit_df)
        summary_frames.append(summary_df)
        for warning in warnings:
            warning_records.append({**warning, "cohort_label": cohort_label})

    per_tumor_fold_results_df = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
    per_tumor_summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
    per_tumor_random_audit_df = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
    loop_warning_df = pd.DataFrame(warning_records)
    overall_summary_df = aggregate_overall_metrics(per_tumor_summary_df) if not per_tumor_summary_df.empty else pd.DataFrame()
    return per_tumor_fold_results_df, per_tumor_summary_df, per_tumor_random_audit_df, loop_warning_df, overall_summary_df



In [ ]:
if run_all_matched_tumor_types:
    (
        per_tumor_fold_results_df,
        per_tumor_summary_df,
        per_tumor_random_audit_df,
        loop_warning_df,
        overall_summary_df,
    ) = run_tumor_type_loop(
        all_samples=all_samples,
        tumor_types=loop_tumor_types,
        cv_n_splits=cv_n_splits,
        meth_data_df=meth_data_df,
        probe_df=probe_df,
        probes_bed=probes_bed,
        base_dir=base_dir,
        genome_file=genome_file,
        top_n_pmd_regions=top_n_pmd_regions,
        random_region_seed=random_region_seed,
        parallelize_folds=parallelize_folds,
        max_fold_workers=max_fold_workers,
        max_feature_workers=max_feature_workers,
    )
else:
    (
        analysis_fold_results_df,
        analysis_random_audit_df,
        analysis_cohort_summary_df,
        analysis_warnings,
        analysis_metadata,
    ) = run_tumor_vs_normal_analysis(
        analysis_samples_df=analysis_samples_df,
        cohort_label=analysis_label,
        tumor_type=tumor_type,
        cv_n_splits=cv_n_splits,
        meth_data_df=meth_data_df,
        probe_df=probe_df,
        probes_bed=probes_bed,
        base_dir=base_dir,
        genome_file=genome_file,
        top_n_pmd_regions=top_n_pmd_regions,
        random_region_seed=random_region_seed,
        parallelize_folds=parallelize_folds,
        max_fold_workers=max_fold_workers,
        max_feature_workers=max_feature_workers,
    )


When `run_all_matched_tumor_types = True`, the plot below uses the aggregated cross-tumor summaries from `overall_summary_df`. `Macro` treats each tumor type equally, while `weighted` gives larger cohorts more influence.

In [ ]:
if run_all_matched_tumor_types:
    plot_loop_overall_summary(overall_summary_df, analysis_label)
    overall_summary_df, per_tumor_summary_df, excluded_projects_df, loop_warning_df.head(30)
else:
    plot_cv_analysis_results(analysis_fold_results_df, analysis_label)
    analysis_cohort_summary_df, analysis_random_audit_df.head(20), pd.DataFrame([analysis_metadata]), pd.DataFrame(analysis_warnings), build_conclusion(analysis_cohort_summary_df)
